# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


In [64]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [65]:
# Load API keys from the .env file in the project root

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AIzaSyAl


In [66]:
# Connect to client libraries.
# Gemini and Ollama both expose an OpenAI-compatible endpoint, so we can reuse
# the OpenAI Python client for all of them - just pointing at a different base_url.

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [67]:
# The two models we'll compare as C++ code generators

OPENAI_MODEL = "gpt-4.1-mini"
GEMINI_MODEL = "gemini-2.5-pro"

In [68]:
# Gather details about this machine (OS, CPU, compiler toolchain) so we can
# tell the LLM exactly what it's compiling for, and ask it to tune accordingly

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Tue Jun  9 22:18:58 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T6000',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M1 Max',
  'cores_logical': 10,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [69]:
# Ask GPT to recommend the fastest clang++ compile command and run command
# for this specific machine - we'll reuse its answer below

message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

You already have Apple clang installed on your Mac (`Apple clang version 21.0.0`), which supports compiling C++ code (it provides `clang++` or `g++` as a frontend).

### Summary:
- You **do NOT need to install any C++ compiler**.
- Apple clang is ready to use out of the box on your M1 Mac.
- You have `make` installed but it’s not strictly necessary for compiling one file.

---

# Instructions to compile and run your `main.cpp` in the simplest way:

You can compile with `clang++` or `g++` (which is an alias to clang++ on your system).

Example command to compile `main.cpp` to executable `main` optimized for fastest runtime:

```sh
clang++ -O3 -std=c++20 -o main main.cpp
```

Explanation:
- `-O3` enables the highest level of optimization for fastest code.
- `-std=c++20` uses C++20 standard (adjust if your code requires another version)
- `-o main` sets the output executable named `main`
- `main.cpp` is your source file

You then run the program with:

```sh
./main
```

---

# Python snippet to compile and run:

```python
import subprocess

compile_command = ["clang++", "-O3", "-std=c++20", "-o", "main", "main.cpp"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["./main"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)
```

---

### If you want a minimal working example for `main.cpp` (for testing only):

```cpp
#include <iostream>

int main() {
    std::cout << "Hello, world!\n";
    return 0;
}
```

---

### Summary:
- No need to install anything
- Use `clang++` from your system
- Compile with optimizations for fastest runtime
- Run the produced executable

Let me know if you want help with something else!

In [70]:
# The compile/run commands GPT recommended above, tuned for max performance on this machine.
# compile_and_run() (defined further down) takes the source filename as an argument,
# so the same compile_command template is reused for every model's C++ output.

compile_command = [
    "clang++", "-std=c++17", "-Ofast",
    "-flto=thin", "-fvisibility=hidden", "-DNDEBUG",
    "main.cpp", "-o", "main"
]
run_command = ["./main"]

## And now, on with the main task

In [71]:
# The prompt that instructs a model to port Python -> fast C++.
# system_prompt sets the ground rules (C++ only, no prose);
# user_prompt_for() adds the machine details and the actual Python source to convert.

system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [72]:
# Build the standard system/user chat messages list for a porting request

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [73]:
# Save generated C++ to its own file (e.g. "openai.cpp", "gemini.cpp") so each
# model's output is kept around for comparison instead of overwriting one main.cpp

def write_output(cpp, filename="main.cpp"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(cpp)

In [74]:
# Send the porting request to a given client/model and return the C++ source.
# Reasoning-capable models get "high" effort; markdown code fences are stripped
# since some models wrap their response in ```cpp ... ``` regardless of instructions.

def port(client, model, python):
    reasoning_effort = "high" if model.startswith(('o1', 'o3', 'o4-mini', 'gpt-5')) else None
    
    kwargs = dict(model=model, messages=messages_for(python))
    if reasoning_effort is not None:
        kwargs["reasoning_effort"] = reasoning_effort
    
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    return reply

In [75]:
# Sample workload: a deliberately slow, pure-Python numeric calculation.
# This is what we'll ask each LLM to port to fast C++, then benchmark against.

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [76]:
# Run the raw Python code directly, to get our baseline (slow) timing

def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [77]:
# Baseline: how long the pure-Python version takes

run_python(pi)

Result: 3.141592656089
Execution Time: 22.153722 seconds


In [78]:
# Ask OpenAI to port the Python code to C++, and save its answer to openai.cpp

cpp = port(openai, OPENAI_MODEL, pi)
write_output(cpp, "openai.cpp")
cpp

'\n#include <cstdio>\n#include <chrono>\n\nint main() {\n    constexpr int iterations = 200\'000\'000;\n    constexpr double param1 = 4.0, param2 = 1.0;\n    double result = 1.0;\n\n    auto start = std::chrono::high_resolution_clock::now();\n\n    for (int i = 1; i <= iterations; ++i) {\n        double j = i * param1 - param2;\n        result -= 1.0 / j;\n        j = i * param1 + param2;\n        result += 1.0 / j;\n    }\n\n    result *= 4.0;\n\n    auto end = std::chrono::high_resolution_clock::now();\n    std::chrono::duration<double> diff = end - start;\n\n    std::printf("Result: %.12f\\n", result);\n    std::printf("Execution Time: %.6f seconds\\n", diff.count());\n\n    return 0;\n}\n'

In [79]:
# Compile the given C++ source file (using the fast-compile flags GPT recommended)
# and run the resulting binary three times, to see timing stabilize once warmed up

def compile_and_run(source_file="main.cpp"):
    command = compile_command[:-3] + [source_file, "-o", "main"]
    subprocess.run(command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [80]:
# Compile and run OpenAI's C++ port

compile_and_run("openai.cpp")

Result: 3.141592656090
Execution Time: 0.146521 seconds

Result: 3.141592656090
Execution Time: 0.132553 seconds

Result: 3.141592656090
Execution Time: 0.145024 seconds



In [81]:
# Speedup factor: Python baseline time / OpenAI C++ execution time

19.178207/0.082168

233.40238292279233

## OK let's try the other contenders!

In [82]:
# Now do the same with Gemini: port to C++, save to gemini.cpp, compile and run

cpp = port(gemini, GEMINI_MODEL, pi)
write_output(cpp, "gemini.cpp")
compile_and_run("gemini.cpp")

Result: 3.141592656200
Execution Time: 0.075554 seconds

Result: 3.141592656200
Execution Time: 0.063469 seconds

Result: 3.141592656200
Execution Time: 0.063504 seconds



In [83]:
# For reference: speedups reported in Ed's original course run (numbers above will
# differ per machine/model version - this is just illustrative, not our own result)

print(f"""
In Ed's experiments, the performance speedups were:

2nd place: GPT-5: {19.178207/0.082168:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")


In Ed's experiments, the performance speedups were:

2nd place: GPT-5: 233X speedup
1st place: Gemini 2.5 Pro: 1440X speedup

